In [0]:
from pyspark.sql import functions as F

# Define the Unity Catalog volume directory containing our JSONL samples.
sample_path = (
    "/Volumes/workspace/amazon_pyspark_backup/"
    "raw_files/electronics_smoke_sample"
)

# Read each JSONL record as text to preserve the original source.
products_text = (
    spark.read.text(f"{sample_path}/products.jsonl")
    .withColumnRenamed("value", "raw_json")
)

# Define the fields needed for this initial inspection.
# Product attributes in "details" are handled separately below.
product_schema = """
    parent_asin STRING,
    title STRING,
    main_category STRING,
    average_rating DOUBLE,
    rating_number BIGINT,
    price STRING,
    store STRING,
    categories ARRAY<STRING>,
    features ARRAY<STRING>,
    description ARRAY<STRING>,
    images ARRAY<STRUCT<
        hi_res: STRING,
        large: STRING,
        thumb: STRING,
        variant: STRING
    >>
"""

# Parse known fields without inferring columns from attribute names.
products_raw = (
    products_text
    .select(
        F.from_json(
            "raw_json",
            product_schema,
            {"mode": "FAILFAST"},
        ).alias("product"),

        # Preserve the attribute object as JSON text.
        F.get_json_object("raw_json", "$.details").alias("details"),

        # Retain every source field for later validation and parsing.
        F.col("raw_json"),
    )
    .select("product.*", "details", "raw_json")
)

In [0]:
# Force parsing of the selected fields and inspect their structure.
products_raw.printSchema()

display(
    products_raw.select(
        "parent_asin",
        "title",
        "price",
        "categories",
        "details",
        "images",
    ).limit(5)
)

print("Product records:", products_raw.count())

In [0]:
# Preview review content, ratings, purchase status, and image structures.
# parent_asin identifies the parent product for a future metadata join.
display(
    reviews_raw.select(
        "parent_asin",
        "rating",
        "title",
        "text",
        "verified_purchase",
        "images",
    ).limit(5)
)

# Preview product attributes to guide schema design and cleaning rules.
display(
    products_raw.select(
        "parent_asin",
        "title",
        "price",
        "categories",
        "features",
        "details",
        "images",
    ).limit(5)
)  